# Step 3: ignore 语法（精确匹配 + `re:` 正则）

**目标**：吃透 `llmcompressor` 的 `ignore` 参数三种写法——(1) 精确层名 `ignore=["lm_head"]`；(2) 正则前缀 `ignore=["re:.*down_proj"]`；(3) 混合。这是把 s1/s2 的「该回退哪些层」**落地成 recipe** 的语法层。M2 只用了精确 ignore（M2.3），正则统一在本节讲透。

**对应 OUTLINE 课时**：3.3 回退机制 ignore 语法（~45 分钟）。


## 学完应能讲清（学完本节应能口头回答）
1. `ignore` 参数接受什么类型？精确写法和正则写法分别长什么样？
2. 正则 ignore **为什么必须加 `re:` 前缀**？不加会怎样？（被当字面字符串，匹配不到任何层）
3. `targets="Linear"` 和 `ignore` 一个按**类名**一个按**层名**，区别是什么？为什么需要两个维度？
4. `re:.*down_proj` 会匹配哪些层？写一个正则只匹配**第 0 层**的所有投影层。（提示：见原理「正则语法速查」表 + 推导——关键是用 `\.` 转义点号。如果你第一次接触正则，先读懂速查表里的 `.`/`\.`/`*` 再写。）
5. `QuantizationModifier`、`GPTQModifier`、`SmoothQuantModifier` 都接受 `targets`+`ignore`——这套统一接口的设计好处是什么？（提示：见原理「为什么 targets+ignore 是统一接口」）

In [ ]:
%%capture
import pathlib, os, re
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()
from transformers import Qwen2Config, Qwen2ForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.modifiers.gptq import GPTQModifier


In [ ]:
# Setup cell（双 env：模块根 = 含 scripts/ + steps/ 的 course/m3-tuning-eval/）。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT      = _find_module_root(pathlib.Path.cwd())
MODEL_DIR        = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"
TINY_MODEL_DIR   = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT         = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT =", MODULE_ROOT, "| 0.5B @", TINY_MODEL_DIR.exists())


## 原理：ignore 三种写法（OUTLINE 3.3）

`ignore` 接受**字符串列表**，元素有两种语义：

| 写法 | 示例 | 匹配规则 |
|---|---|---|
**精确匹配** | `ignore=["lm_head"]` | 层名**完全等于**该字符串 |
**正则匹配** | `ignore=["re:.*down_proj"]` | 去掉 `re:` 后，层名 `re.search` 命中即匹配 |
**混合** | `ignore=["lm_head", "re:.*down_proj"]` | 两类同时生效 |

**关键区分**（OUTLINE 易错点）：
- `targets="Linear"` 是**类名匹配**（所有 `nn.Linear` 实例）。
- `ignore=[...]` 是**层名匹配**（按模块的 `name`，如 `model.layers.0.mlp.down_proj`）。
- 正则**不加 `re:` 前缀不生效**——会被当成字面字符串去精确匹配（几乎匹配不到任何层）。

正则的 `re:` 前缀语义：`llmcompressor` 看到 `re:xxx` 就把 `xxx` 当 Python `re` 正则，对每个候选层名做 `re.search`（不是 fullmatch，是子串/模式搜索）。

### 正则语法速查（第 4 题要你「自己写正则」，先认得这些符号）

零基础第一次写正则，认得这几个符号就够了——它们是「描述一类字符串」的小语言：

| 符号 | 含义 | 在层名里怎么用 |
|---|---|---|
`.` | 通配**任意一个字符**（一个，不是零个） | `layers.0` 里的 `.` 在正则里也是通配，要匹配字面点号得转义 |
`\.` | 字面点号（`.` 前加 `\` 转义，表示「我就要这个点」） | 匹配 `model.layers.0` 的点号要写 `model\.layers\.0` |
`*` | 前面的东西重复 **0 次或多次** | `.*` = 任意长度任意字符（最常用的「随便什么前缀/后缀」） |
`+` | 前面的东西重复 **1 次或多次** | 类似 `*` 但至少要有一次 |
`\d` | 任意一个数字（0-9） | `\d` 可匹配层号，如 `layers.\d` 匹配 `layers.0`、`layers.3` |
`\|` | 或（左右任一命中即可） | `down_proj\|gate_proj` 匹配两种后缀 |
`^` | 锚定**开头** | `^model` 要求层名必须以 model 开头（search 下通常可省） |

**推导两个实战正则**（对应第 4 题）：
- `re:.*down_proj` → 去掉 `re:` 后是 `.*down_proj`。`.*` 匹配任意前缀、`down_proj` 是字面（这些字母不是正则元字符、不用转义）。`re.search` 在 `model.layers.0.mlp.down_proj` 里找到末尾的 `down_proj` → 命中。所以它匹配**所有层的 `down_proj`**（每层都有）。
- 只匹配**第 0 层**所有投影层：层名形如 `model.layers.0.mlp.down_proj`、`model.layers.0.self_attn.q_proj`。第 0 层的共同特征是 `layers.0.` 这一段。正则要写成 `.*layers\.0\..*`（注意：`0` 前后的两个点号都要 `\.` 转义成字面点，否则 `.` 当通配会连 `layers101` 这种也匹配上；外层 `.*` 吃掉 `model.` 前缀和 `.mlp.down_proj` 后缀）。加 `re:` 前缀 → **`re:.*layers\.0\..*`**。用 `re.search` 对每个层名跑一遍，只有第 0 层的层名命中。

> 提示：本节填空 `build_regex_ignore` 用了 Python 的 `re.escape(suffix)`——它自动把 suffix 里的所有正则元字符（如 `.`）转义掉。所以当你只想匹配「一段字面后缀」、又懒得手动转义时，`.*` + `re.escape(suffix)` 拼接是最稳的写法（见本步填空 1）。

### 为什么 targets+ignore 是「统一接口」（第 5 题）

`QuantizationModifier`、`GPTQModifier`、`SmoothQuantModifier` 三个 modifier 都吃同一套 `targets`+`ignore`，**这不是巧合而是刻意的接口设计**，好处有四：

1. **学一套语法表达任意「哪些层量化/哪些回退」的意图**：不管你用朴素量化、GPTQ 还是 SmoothQuant，描述「目标层/排除层」的方式完全一样——只学一次 s3，三个算法都能调。
2. **跨 modifier 可复用**：s1/s5 扫出的敏感层清单（同一组 `ignore` 正则），可以原样喂给任一 modifier，换算法不用换 ignore 语法。
3. **降低学习与迁移成本**：从 `QuantizationModifier`（M2）升级到 `GPTQModifier`/`SmoothQuantModifier`（M3），不需要重学「怎么指定层」，只学算法本身的差异。
4. **一套语法管多个算法 = 可组合**：s4 的 mixed-precision 能把多个 modifier 叠在一次 oneshot 里，正是因为它们共享同一套 targets+ignore 语义，llmcompressor 才能把多个「目标层/排除层」声明合并成一份 `config_groups`。

一句话：统一接口是「调优意图（哪些层量化/哪些回退）」和「执行（具体量化算法）」之间的**唯一解耦层**——意图用一套语法表达，算法在底下换，互不干扰。

### 端到端：ignore 在 layer fallback 全流程的位置

s1 给敏感度排序 → s2 给经验法则清单 → **s3 把这些清单翻译成 `ignore` 语法** → 喂给 `QuantizationModifier`/`GPTQModifier` 跑 oneshot。ignore 是「调优意图」和「llmcompressor 执行」之间的唯一接口。s4 的 mixed-precision 是它的进阶（不同层不同 scheme，但语法同源）。


## 亲手摸一摸：精确 vs 正则 ignore 命中哪些层

先建 tiny 模型，用 Python 正则模拟 `re:` 语义，看不同 ignore 写法命中哪些层名——直观理解「精确」与「正则」的覆盖差异。


In [ ]:
## 摸一摸：模拟 llmcompressor 的 ignore 匹配逻辑
tiny = Qwen2ForCausalLM(Qwen2Config(
    num_hidden_layers=4, hidden_size=64, intermediate_size=128,
    num_attention_heads=4, num_key_value_heads=2, vocab_size=320, tie_word_embeddings=False))
linear_names = [n for n, m in tiny.named_modules() if isinstance(m, nn.Linear)]

def matches_ignore(name, ignore):
    """模拟 llmcompressor ignore 匹配：精确 == 或 re: 前缀正则 search。"""
    for pat in ignore:
        if pat.startswith("re:"):
            if re.search(pat[3:], name): return True
        else:
            if name == pat: return True
    return False

print("=== 精确 ignore=['lm_head'] 命中 ===")
print([n for n in linear_names if matches_ignore(n, ['lm_head'])])
print("\n=== 正则 ignore=['re:.*down_proj'] 命中（所有 down_proj）===")
print([n for n in linear_names if matches_ignore(n, ['re:.*down_proj'])])
print("\n=== 正则 ignore=['re:.*layers\\.0\\..*'] 命中（第 0 层全部）===")
print([n for n in linear_names if matches_ignore(n, ['re:.*layers\\.0\\..*'])])
print("\n=== 漏写 re: 前缀（'down_proj' 当字面）命中 ===")
print([n for n in linear_names if matches_ignore(n, ['down_proj'])], "← 几乎匹配不到（易错点）")


## 本步填空

1. **`build_regex_ignore(suffixes)`** —— 把一组层类后缀（如 `["down_proj", "gate_proj"]`）转成一条 `re:.*down_proj|.*gate_proj` 风格的正则 ignore 项（一条管多个类，比每类一条 ignore 简洁）。**为什么这么设计（填前先想）**：多个相关层共用一个 ignore 正则，recipe 更短、可读性更好。
2. **`is_layer_ignored(name, ignore)`**（判断型）—— 给定层名和 ignore 列表，返回该层是否会被 ignore（精确 == 或 `re:` 正则 search）。**为什么这么设计**：调优时要快速核验「我这条 ignore 到底会不会命中某层」——封装成判断函数，既可单元测试，也是 L2 验证「lm_head 真被回退」的工具。


In [ ]:
def build_regex_ignore(suffixes):
    """把一组层类后缀转成一条合并的正则 ignore 项（带 're:' 前缀）。
    例 build_regex_ignore(['down_proj', 'gate_proj']) -> 're:.*down_proj|.*gate_proj'。

    为什么这么设计（填前先想）：多个相关层共用一条 ignore 正则比每类一条更简洁、
    recipe 可读性更好；用 | 合并 .*suffix 模式，re.search 命中任一后缀即匹配。
    """
    # TODO: 1) 对每个 suffix 构造 '.*' + re.escape(suffix)（escape 防止 suffix 含正则元字符）。
    #       2) 用 '|' join 成一条，前面加 're:'。
    #       3) 返回该字符串。
    raise NotImplementedError


In [ ]:
# 参考实现（reviewer 注入执行验证用；学员勿看）
def _build_regex_ignore_ref(suffixes):
    if not suffixes:
        raise ValueError("suffixes 不能为空")
    body = "|".join(".*" + re.escape(s) for s in suffixes)
    return "re:" + body
build_regex_ignore = _build_regex_ignore_ref  # noqa


In [ ]:
def is_layer_ignored(name, ignore):
    """判断型：给定层名 name 和 ignore 列表，返回该层是否会被 ignore（True=回退）。
    匹配语义同 llmcompressor：
      - 're:xxx' -> re.search(xxx, name) 命中即 True。
      - 其它 -> name == pat 精确相等才 True。

    为什么这么设计（填前先想）：调优要快速核验 ignore 是否真命中目标层（防写错正则静默失效）。
    封装成判断函数可单元测试，也是 L2 验证「lm_head 真回退」的工具。
    """
    # TODO: 遍历 ignore 每项 pat：
    #   - 若 pat.startswith('re:')：re.search(pat[3:], name) 命中即 return True。
    #   - 否则：name == pat 即 return True。
    #   遍历完无命中 return False。
    raise NotImplementedError


In [ ]:
# 参考实现（reviewer 注入执行验证用；学员勿看）
def _is_layer_ignored_ref(name, ignore):
    for pat in ignore:
        if pat.startswith("re:"):
            if re.search(pat[3:], name):
                return True
        elif name == pat:
            return True
    return False
is_layer_ignored = _is_layer_ignored_ref  # noqa


In [ ]:
def _names():
    m = Qwen2ForCausalLM(Qwen2Config(
        num_hidden_layers=2, hidden_size=64, intermediate_size=128,
        num_attention_heads=4, num_key_value_heads=2, vocab_size=320, tie_word_embeddings=False))
    return [n for n, mod in m.named_modules() if isinstance(mod, nn.Linear)]

def test_build_regex_ignore_single_and_multi():
    assert build_regex_ignore(["down_proj"]) == "re:.*down_proj"
    out = build_regex_ignore(["down_proj", "gate_proj"])
    assert out.startswith("re:")
    body = out[3:]
    assert ".*down_proj" in body and ".*gate_proj" in body and "|" in body

def test_build_regex_ignore_escapes_metachars():
    out = build_regex_ignore(["a.b"])
    assert "a\\.b" in out, "点号应被 re.escape 转义"

def test_is_layer_ignored_exact_match():
    assert is_layer_ignored("lm_head", ["lm_head"]) is True
    assert is_layer_ignored("model.layers.0.self_attn.q_proj", ["lm_head"]) is False

def test_is_layer_ignored_regex_match():
    ignore = ["lm_head", "re:.*down_proj"]
    assert is_layer_ignored("model.layers.0.mlp.down_proj", ignore) is True
    assert is_layer_ignored("model.layers.0.self_attn.q_proj", ignore) is False
    assert is_layer_ignored("lm_head", ignore) is True  # 精确项也命中

def test_is_layer_ignored_missing_re_prefix_is_literal():
    # 漏写 re: 前缀 -> 当字面，不匹配（易错点验证）
    assert is_layer_ignored("model.layers.0.mlp.down_proj", ["down_proj"]) is False

# L1 必过守卫：ipytest.run 返回 pytest 退出码；非 0（有测试失败）→ 抛异常让 nbconvert 真挂。
# （用 ipytest.run() 而非 %%ipytest magic：magic 吞掉失败、exit_code 属性在本版不可靠。）
_ec = ipytest.run("-qq")
assert _ec == 0, f"L1 测试未全过（exit_code={_ec}），见上方 pytest 输出。"

## L2（tiny，CPU）：ignore 真跑通语义

用学员的 `is_layer_ignored` + `build_regex_ignore` 构造 ignore，验证被 ignore 的层确实是目标层、没被 ignore 的层会被量化。


In [ ]:
## L2：tiny ignore 语义验证——被 ignore 的层保持 FP16
tiny = Qwen2ForCausalLM(Qwen2Config(
    num_hidden_layers=2, hidden_size=64, intermediate_size=128,
    num_attention_heads=4, num_key_value_heads=2, vocab_size=320, tie_word_embeddings=False))
linear_names = [n for n, m in tiny.named_modules() if isinstance(m, nn.Linear)]
ignore = ["lm_head", build_regex_ignore(["down_proj"])]
print("构造的 ignore =", ignore)

ignored = [n for n in linear_names if is_layer_ignored(n, ignore)]
quantized = [n for n in linear_names if not is_layer_ignored(n, ignore)]
print(f"\n将被 ignore（保持 FP16）的层 ({len(ignored)})：")
for n in ignored: print(f"  - {n}")
print(f"将被量化（W8A8）的层 ({len(quantized)})：")
for n in quantized: print(f"  + {n}")

assert is_layer_ignored("lm_head", ignore), "lm_head 应被精确 ignore"
assert is_layer_ignored("model.layers.0.mlp.down_proj", ignore), "regex 应命中 down_proj"
assert not is_layer_ignored("model.layers.0.self_attn.q_proj", ignore), "q_proj 不该被 ignore"
assert all("down_proj" in n for n in ignored if n != "lm_head"), "regex 应只命中 down_proj"
assert all("down_proj" not in n and n != "lm_head" for n in quantized), "其余层应被量化"
print("\nL2 通过：ignore 精确+正则语义正确，lm_head 与所有 down_proj 被回退。")


## L3（H200，GPU 守卫）：真 7B ignore 前后 PPL 对比

在 7B 上跑 SmoothQuant W8A8 两次：一次全量化（无 ignore）、一次 ignore 掉 s1/s2 找出的敏感层。对比 PPL——验证 ignore 真的能救回精度。


In [ ]:
import torch, os
def run_l3_ignore_ppl():
    from llmcompressor.modifiers.transform.smoothquant import SmoothQuantModifier
    from datasets import load_dataset
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    calib = load_dataset("wikitext", "wikitext-2-raw-v1", split="train").shuffle(seed=0)["text"][:32]

    def run(tag, ignore):
        recipe = [SmoothQuantModifier(smoothing_strength=0.8),
                  GPTQModifier(targets="Linear", scheme="W8A8", ignore=ignore or ["lm_head"])]
        out = OUT_ROOT / f"s3_{tag}"
        oneshot(model=str(MODEL_DIR), tokenizer=str(MODEL_DIR), recipe=recipe,
                dataset=calib, num_calibration_samples=32, output_dir=str(out))
        return out

    import json
    rank = json.loads((OUT_ROOT / "s1_sensitive_rank.json").read_text()) if (OUT_ROOT / "s1_sensitive_rank.json").exists() else []
    sensitive_suffixes = list({r[0].split(".")[-1] for r in rank if r[0] != "lm_head"})[:4]
    ignore = ["lm_head"] + ([build_regex_ignore(sensitive_suffixes)] if sensitive_suffixes else [])
    print("调优 ignore =", ignore)
    out_all = run("all_quant", None)
    out_ig  = run("ignore", ignore)
    print(f"产物：全量化 @ {out_all} | ignore 调优 @ {out_ig}")

if torch.cuda.is_available() and not os.environ.get("SKIP_L3"):
    run_l3_ignore_ppl()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（CPU/CI 只验 L1+L2 ignore 语法逻辑）")


## 产物检查


In [ ]:
import json
for tag in ["all_quant", "ignore"]:
    p = OUT_ROOT / f"s3_{tag}" / "config.json"
    if p.exists():
        qc = json.loads(p.read_text()).get("quantization_config", {})
        groups = qc.get("config_groups", {})
        g0 = list(groups.values())[0] if groups else {}
        print(f"s3_{tag}: ignore={qc.get('ignore', g0.get('ignore'))} | targets={g0.get('targets')}")
    else:
        print(f"s3_{tag}: 产物不存在（L3 未跑或被 SKIP_L3 跳过）")
